# 🎬 AIC 2026 - Benchmark Bóc Tách Keyframe (PySceneDetect + N = 2*sqrt(t))

Notebook này phục vụ mục đích **đo đạc hiệu năng thực tế (Benchmark)** trên môi trường **Kaggle** để so sánh tốc độ với **Máy ảo (VM Linux venv)** khi xử lý bóc tách Keyframe cho tập dữ liệu Batch 2 (~250GB video thô).

### 🎯 Chiến thuật bóc tách được kiểm thử:
1. **Phát hiện cắt cảnh:** Dùng `PySceneDetect` (`ContentDetector`) quét toàn bộ frame video.
2. **Quy tắc trích xuất:** Mỗi phân cảnh (scene) dài $t$ giây sẽ trích xuất $N = \max(1, \text{round}(2 \times \sqrt{t}))$ keyframes.
3. **Ghi đĩa:** Trích xuất frame bằng `cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)` và lưu ảnh `.jpg` vào `/kaggle/working`.

--- 
## BƯỚC 1: Kiểm Tra Tài Nguyên Phần Cứng Kaggle (CPU, RAM, Ổ Cứng)

In [ ]:
import os
import shutil
import platform

print("=" * 65)
print("🖥️  THÔNG SỐ PHẦN CỨNG MÔI TRƯỜNG KAGGLE")
print("=" * 65)
# Kiểm tra CPU cores
cpu_count = os.cpu_count()
print(f"🔹 Số lượng vCPU Cores : {cpu_count} vCPU ({platform.processor() or platform.machine()})")

# Kiểm tra RAM
try:
    with open('/proc/meminfo', 'r') as f:
        mem_lines = f.readlines()
    mem_total = int([x for x in mem_lines if 'MemTotal' in x][0].split()[1]) / (1024**2)
    mem_avail = int([x for x in mem_lines if 'MemAvailable' in x][0].split()[1]) / (1024**2)
    print(f"🔹 RAM Khả dụng        : {mem_avail:.2f} GB / {mem_total:.2f} GB")
except Exception:
    pass

# Kiểm tra dung lượng ổ đĩa /kaggle/working
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"🔹 Dung lượng ổ đĩa     : Đã dùng {used/(1024**3):.2f} GB | Trống {free/(1024**3):.2f} GB / {total/(1024**3):.2f} GB")
print("=" * 65)

--- 
## BƯỚC 2: Cài Đặt Thư Viện Cần Thiết

In [ ]:
!pip install -q scenedetect opencv-python-headless psutil tqdm
print("✅ Đã cài đặt xong thư viện scenedetect, opencv-python-headless và psutil!")

--- 
## BƯỚC 3: Cấu Hình Video Test & Thư Mục Xuất Keyframe

In [ ]:
import glob

# ---------------------------------------------------------------------------
# ĐẠI CA ĐIỀN ĐƯỜNG DẪN VIDEO TEST CẦN BENCHMARK TẠI ĐÂY (HOẶC ĐỂ TỰ ĐỘNG TÌM)
# ---------------------------------------------------------------------------
VIDEO_PATH = "" # Ví dụ: "/kaggle/input/aic2026-batch2/video/L21_V001.mp4"
OUTPUT_DIR = "/kaggle/working/benchmark_kf"
DETECTOR_THRESHOLD = 27.0 # Ngưỡng cắt cảnh tương đồng với file Slurm

# Tự động phát hiện video trong /kaggle/input nếu đại ca chưa điền path
if not VIDEO_PATH or not os.path.exists(VIDEO_PATH):
    found_videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
    if found_videos:
        VIDEO_PATH = found_videos[0]
        print(f"🎯 Tự động chọn video test đầu tiên: {VIDEO_PATH}")
    else:
        print("⚠️ Không tìm thấy file .mp4 nào trong /kaggle/input!")
        print("👉 Đại ca hãy gán đường dẫn video cụ thể vào biến VIDEO_PATH ở trên.")
else:
    print(f"🎯 Video test được chọn: {VIDEO_PATH}")

--- 
## BƯỚC 4: Chạy Benchmark Bóc Tách Keyframe & Đo Lường Chi Tiết

In [ ]:
import time
import math
import cv2
from scenedetect import detect, ContentDetector

def get_num_keyframes(duration_sec: float) -> int:
    """Hàm Toán học căn bậc hai: N = 2 * sqrt(t) để chống bùng nổ dữ liệu"""
    if duration_sec <= 0:
        return 1
    return max(1, int(round(2 * math.sqrt(duration_sec))))

def run_benchmark(video_path: str, output_dir: str, threshold: float = 27.0):
    if not os.path.exists(video_path):
        print(f"❌ File video không tồn tại: {video_path}")
        return
        
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    vid_out_dir = os.path.join(output_dir, video_name)
    os.makedirs(vid_out_dir, exist_ok=True)

    # Đọc metadata video
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_sec = total_frames / fps if fps > 0 else 0
    cap.release()
    file_size_mb = os.path.getsize(video_path) / (1024 * 1024)

    print("=" * 70)
    print("🚀 BẮT ĐẦU BENCHMARK TRÍCH XUẤT KEYFRAME TRÊN KAGGLE")
    print("=" * 70)
    print(f"📁 Video Path        : {video_path}")
    print(f"📏 Dung lượng file   : {file_size_mb:.2f} MB")
    print(f"⏱️  Thời lượng video : {duration_sec:.2f}s ({duration_sec/60:.2f} phút)")
    print(f"🎞️  Tổng số frames   : {total_frames} frames @ {fps:.2f} FPS ({width}x{height})")
    print("-" * 70)

    # ---------------------------------------------------------
    # GIAI ĐOẠN 1: Phát hiện cắt cảnh (PySceneDetect)
    # ---------------------------------------------------------
    print("🔍 [GIAI ĐOẠN 1/2] Đang quét cắt cảnh (PySceneDetect ContentDetector)...\n")
    t0_detect = time.perf_counter()
    scene_list = detect(video_path, ContentDetector(threshold=threshold))
    t1_detect = time.perf_counter()
    detect_duration = t1_detect - t0_detect
    detect_fps = total_frames / detect_duration if detect_duration > 0 else 0

    print(f"✅ Giai đoạn 1 hoàn tất:")
    print(f"   - Số cảnh phát hiện  : {len(scene_list)} scenes")
    print(f"   - Thời gian detect   : {detect_duration:.2f}s")
    print(f"   - Tốc độ quét scene  : {detect_fps:.2f} video frames/s ({detect_fps/fps:.2f}x Real-time)\n")

    # ---------------------------------------------------------
    # GIAI ĐOẠN 2: Seek frame & Ghi đĩa (OpenCV cv2.imwrite)
    # ---------------------------------------------------------
    print("📸 [GIAI ĐOẠN 2/2] Đang trích xuất frame & lưu ảnh JPG vào /kaggle/working...")
    t0_extract = time.perf_counter()
    
    cap = cv2.VideoCapture(video_path)
    extracted_count = 0
    planned_frames = []

    for scene in scene_list:
        start_frame = scene[0].get_frames()
        end_frame = scene[1].get_frames()
        sc_duration = (end_frame - start_frame) / fps
        num_kf = get_num_keyframes(sc_duration)

        if num_kf == 1:
            targets = [start_frame]
        else:
            step = (end_frame - start_frame - 1) / (num_kf - 1)
            targets = [int(start_frame + step * j) for j in range(num_kf)]
        planned_frames.extend(targets)

    print(f"   - Tổng số KF dự kiến trích xuất: {len(planned_frames)} frames")

    for idx, f_idx in enumerate(planned_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
        ret, frame = cap.read()
        if ret:
            out_path = os.path.join(vid_out_dir, f"{f_idx:06d}.jpg")
            cv2.imwrite(out_path, frame)
            extracted_count += 1

        if (idx + 1) % 50 == 0 or (idx + 1) == len(planned_frames):
            progress_pct = (idx + 1) / len(planned_frames) * 100
            elapsed_sec = time.perf_counter() - t0_extract
            eta_sec = (elapsed_sec / (idx + 1)) * (len(planned_frames) - (idx + 1))
            print(f"   [Tiến độ] {idx + 1}/{len(planned_frames)} ({progress_pct:.1f}%) | Đã chạy: {elapsed_sec:.1f}s | ETA: {eta_sec:.1f}s")

    cap.release()
    t1_extract = time.perf_counter()
    extract_duration = t1_extract - t0_extract
    write_fps = extracted_count / extract_duration if extract_duration > 0 else 0

    total_time = detect_duration + extract_duration
    overall_fps = total_frames / total_time if total_time > 0 else 0
    speedup_factor = duration_sec / total_time if total_time > 0 else 0

    # Tính dung lượng thư mục ảnh vừa sinh ra
    total_bytes = sum(os.path.getsize(os.path.join(vid_out_dir, f)) for f in os.listdir(vid_out_dir) if f.endswith('.jpg'))
    total_size_mb = total_bytes / (1024 * 1024)

    # ---------------------------------------------------------
    # BẢNG TỔNG KẾT BENCHMARK
    # ---------------------------------------------------------
    print("\n" + "=" * 70)
    print("📊 BẢNG TỔNG KẾT HIỆU NĂNG TRÊN KAGGLE")
    print("=" * 70)
    print(f"1. Quét PySceneDetect       : {detect_duration:.2f}s ({detect_duration/total_time*100:.1f}%) | {detect_fps:.1f} video frames/s")
    print(f"2. Seek & Lưu ảnh đĩa (/working) : {extract_duration:.2f}s ({extract_duration/total_time*100:.1f}%) | {write_fps:.1f} ảnh/s")
    print(f"3. TỔNG THỜI GIAN THỰC THI  : {total_time:.2f}s ({total_time/60:.2f} phút)")
    print(f"4. Tốc độ trung bình toàn bộ: {overall_fps:.1f} frames/giây")
    print(f"5. Tỷ lệ xử lý so với video : {speedup_factor:.2f}x (Video {duration_sec:.1f}s xong trong {total_time:.1f}s)")
    print(f"6. Số keyframes sinh ra     : {extracted_count} ảnh | {total_size_mb:.2f} MB (~{total_size_mb/max(1, extracted_count)*1024:.1f} KB/ảnh)")
    print("=" * 70)

if VIDEO_PATH and os.path.exists(VIDEO_PATH):
    run_benchmark(VIDEO_PATH, OUTPUT_DIR, DETECTOR_THRESHOLD)
else:
    print("⚠️ Vui lòng cung cấp VIDEO_PATH hợp lệ để bắt đầu benchmark!")

--- 
## BƯỚC 5 (Tùy Chọn): Dọn Dẹp Ảnh Test Sau Khi Benchmark
Vì ổ cứng `/kaggle/working` chỉ có ~20GB, sau khi xem xong số liệu benchmark, đại ca có thể chạy cell này để dọn dẹp thư mục ảnh test.

In [ ]:
# Dọn dẹp thư mục test để giải phóng dung lượng đĩa
import shutil

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print(f"🧹 Đã xóa dọn dẹp sạch thư mục: {OUTPUT_DIR}")
else:
    print("Thư mục không tồn tại hoặc đã được dọn dẹp.")